# ДЗ 2 — Исследование фильтрации изображений

**Цель:** зашумить изображение гауссовским и постоянным шумом, протестировать медианный, гауссовский, билатеральный фильтры и фильтр нелокальных средних с разными параметрами, а затем определить лучший результат по количественным метрикам.

> **Как пользоваться:** запустите ячейки сверху вниз. В ячейке загрузки можно указать путь к своему изображению. Если путь оставить пустым, будет создано тестовое изображение, чтобы ноутбук запускался без дополнительных файлов.


In [ ]:
# Импорт библиотек
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["image.cmap"] = "gray"


## 1. Загрузка исходного изображения

In [ ]:
# Укажите путь к своему изображению, например:
# IMAGE_PATH = "my_image.jpg"
# Если оставить пустым, создастся тестовое изображение.

IMAGE_PATH = ""

def make_test_image(h=512, w=768):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    # градиент
    x = np.linspace(0, 255, w, dtype=np.uint8)
    y = np.linspace(0, 255, h, dtype=np.uint8)
    img[:, :, 0] = x[None, :]
    img[:, :, 1] = y[:, None]
    img[:, :, 2] = 120

    # геометрические объекты
    cv2.rectangle(img, (60, 60), (280, 220), (240, 240, 240), -1)
    cv2.circle(img, (500, 180), 100, (30, 180, 70), -1)
    cv2.line(img, (80, 400), (680, 300), (255, 255, 255), 8)
    cv2.putText(img, "TEST", (250, 430), cv2.FONT_HERSHEY_SIMPLEX,
                2.2, (20, 20, 20), 6, cv2.LINE_AA)
    return img

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    original_bgr = cv2.imread(IMAGE_PATH)
    if original_bgr is None:
        raise ValueError("Не удалось прочитать изображение.")
    original = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
    print("Загружено:", IMAGE_PATH)
else:
    original = make_test_image()
    print("Используется тестовое изображение.")

# Для эксперимента приводим изображение к размеру не больше 900 px по большей стороне
max_side = 900
scale = min(1.0, max_side / max(original.shape[:2]))
if scale < 1:
    original = cv2.resize(
        original, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA
    )

original = original.astype(np.uint8)

plt.figure(figsize=(12, 7))
plt.imshow(original)
plt.title("Исходное изображение")
plt.axis("off")
plt.show()


## 2. Добавление шума

In [ ]:
def add_gaussian_noise(image, sigma=25, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_constant_noise(image, value=35):
    # Постоянный аддитивный шум: одно и то же смещение яркости для всех пикселей.
    noisy = image.astype(np.float32) + value
    return np.clip(noisy, 0, 255).astype(np.uint8)

gaussian_noisy = add_gaussian_noise(original, sigma=25)
constant_noisy = add_constant_noise(original, value=35)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original)
axes[0].set_title("Исходное")
axes[1].imshow(gaussian_noisy)
axes[1].set_title("Гауссовский шум, σ=25")
axes[2].imshow(constant_noisy)
axes[2].set_title("Постоянный шум, +35")
for ax in axes:
    ax.axis("off")
plt.show()


## 3. Реализация фильтров

Проверяются:
- **медианный фильтр** — разные размеры окна;
- **гауссовский фильтр** — разные σ;
- **билатеральный фильтр** — разные параметры пространственной и цветовой близости;
- **нелокальные средние (NLM)** — разные значения `h`.

Для каждого результата считаются **PSNR** и **SSIM** относительно исходного изображения. Чем выше эти значения, тем ближе отфильтрованное изображение к исходному.


In [ ]:
def to_bgr(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

def to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def median_filter(img, k):
    return cv2.medianBlur(img, k)

def gaussian_filter(img, sigma):
    # Размер ядра выбираем автоматически из sigma
    k = max(3, int(2 * round(3 * sigma) + 1))
    return cv2.GaussianBlur(img, (k, k), sigmaX=sigma, sigmaY=sigma)

def bilateral_filter(img, d, sigma_color, sigma_space):
    return cv2.bilateralFilter(img, d, sigma_color, sigma_space)

def nlm_filter(img, h):
    return cv2.fastNlMeansDenoisingColored(
        img, None, h, h, templateWindowSize=7, searchWindowSize=21
    )

def metrics(reference, result):
    psnr = peak_signal_noise_ratio(reference, result, data_range=255)
    ssim = structural_similarity(reference, result, channel_axis=2, data_range=255)
    return psnr, ssim


In [ ]:
def run_experiment(noisy, noise_name):
    noisy_bgr = to_bgr(noisy)
    rows = []

    # Медианный
    for k in [3, 5, 7]:
        out = median_filter(noisy, k)
        psnr, ssim = metrics(original, out)
        rows.append(["Медианный", f"k={k}", noise_name, psnr, ssim, out])

    # Гауссовский
    for sigma in [0.8, 1.5, 2.5]:
        out = gaussian_filter(noisy, sigma)
        psnr, ssim = metrics(original, out)
        rows.append(["Гауссовский", f"σ={sigma}", noise_name, psnr, ssim, out])

    # Билатеральный
    for d, sc, ss in [(5, 30, 30), (7, 50, 50), (9, 80, 80)]:
        out_bgr = bilateral_filter(noisy_bgr, d, sc, ss)
        out = to_rgb(out_bgr)
        psnr, ssim = metrics(original, out)
        rows.append(["Билатеральный", f"d={d}, σc={sc}, σs={ss}", noise_name, psnr, ssim, out])

    # NLM
    for h in [5, 10, 15, 25]:
        out_bgr = nlm_filter(noisy_bgr, h)
        out = to_rgb(out_bgr)
        psnr, ssim = metrics(original, out)
        rows.append(["Нелокальные средние", f"h={h}", noise_name, psnr, ssim, out])

    return rows

all_rows = []
all_rows.extend(run_experiment(gaussian_noisy, "Гауссовский"))
all_rows.extend(run_experiment(constant_noisy, "Постоянный"))

results = pd.DataFrame(
    all_rows,
    columns=["Фильтр", "Параметры", "Тип шума", "PSNR, dB", "SSIM", "Изображение"]
)

display(
    results.drop(columns=["Изображение"])
    .sort_values(["Тип шума", "PSNR, dB"], ascending=[True, False])
    .reset_index(drop=True)
)


## 4. Сравнение результатов

In [ ]:
# Для каждого типа шума показываем лучшие результаты по PSNR и SSIM.

for noise_name in ["Гауссовский", "Постоянный"]:
    subset = results[results["Тип шума"] == noise_name].copy()
    best_psnr = subset.loc[subset["PSNR, dB"].idxmax()]
    best_ssim = subset.loc[subset["SSIM"].idxmax()]

    print(f"\n=== {noise_name} шум ===")
    print("Лучший по PSNR:")
    print(f"  {best_psnr['Фильтр']} ({best_psnr['Параметры']}): "
          f"PSNR={best_psnr['PSNR, dB']:.2f} dB, SSIM={best_psnr['SSIM']:.4f}")
    print("Лучший по SSIM:")
    print(f"  {best_ssim['Фильтр']} ({best_ssim['Параметры']}): "
          f"PSNR={best_ssim['PSNR, dB']:.2f} dB, SSIM={best_ssim['SSIM']:.4f}")


In [ ]:
# Визуализация лучших вариантов для каждого типа шума по PSNR

for noise_name in ["Гауссовский", "Постоянный"]:
    subset = results[results["Тип шума"] == noise_name]
    best = subset.loc[subset["PSNR, dB"].idxmax()]

    noisy = gaussian_noisy if noise_name == "Гауссовский" else constant_noisy

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original)
    axes[0].set_title("Исходное")
    axes[1].imshow(noisy)
    axes[1].set_title(f"{noise_name} шум")
    axes[2].imshow(best["Изображение"])
    axes[2].set_title(
        f"Лучший: {best['Фильтр']}\n{best['Параметры']}\n"
        f"PSNR={best['PSNR, dB']:.2f} dB, SSIM={best['SSIM']:.4f}"
    )
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Средние результаты по типу фильтра.
# Это помогает сравнить не отдельный набор параметров, а семейства методов.

summary = (
    results
    .groupby(["Тип шума", "Фильтр"])[["PSNR, dB", "SSIM"]]
    .mean()
    .reset_index()
)

display(summary.sort_values(["Тип шума", "PSNR, dB"], ascending=[True, False]))


## 5. Вывод

В качестве основного критерия используем **PSNR**, а **SSIM** — как дополнительный показатель сохранения структуры изображения.

После запуска ноутбук автоматически выводит:
1. результаты всех проверенных параметров;
2. лучший вариант для гауссовского шума;
3. лучший вариант для постоянного шума;
4. визуальное сравнение;
5. средние показатели фильтров.

**Важно:** лучший фильтр зависит от типа и силы шума. Поэтому в отчёте следует ориентироваться на фактические значения PSNR/SSIM, полученные на используемом изображении, а не заранее утверждать, что один метод всегда лучше остальных.
